<a href="https://colab.research.google.com/github/khalidkhankakar/Hands-on-Machine-Learning/blob/master/chapter_10_ANN/neural_network_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Linear Reqression with Pytorch

In [66]:
import torch
from sklearn.datasets import fetch_california_housing
X, y = fetch_california_housing(return_X_y=True)
X.shape, y.shape


((20640, 8), (20640,))

In [67]:
from sklearn.model_selection import train_test_split
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, random_state=42)
X_train, X_dev, y_train, y_dev = train_test_split(X_train_full, y_train_full, random_state=42)
X_train.shape, y_train.shape, X_test.shape, y_test.shape, X_dev.shape, y_dev.shape

((11610, 8), (11610,), (5160, 8), (5160,), (3870, 8), (3870,))

In [68]:
X_train = torch.FloatTensor(X_train)
X_dev = torch.FloatTensor(X_dev)
X_test = torch.FloatTensor(X_test)
means = X_train.mean(dim=0, keepdim=True)
stds = X_train.std(dim=0, keepdim=True)
X_train = (X_train - means) / stds
X_test = (X_test- means) / stds
X_dev = (X_dev - means) / stds

In [69]:
y_train= torch.FloatTensor(y_train.reshape(-1, 1))
y_dev= torch.FloatTensor(y_dev.reshape(-1, 1))
y_test= torch.FloatTensor(y_test.reshape(-1, 1))

In [70]:
# Our parameter for network
n_features = X_train.shape[1]
weights = torch.randn((n_features, 1), requires_grad=True)
bias = torch.tensor(0., requires_grad=True)

In [71]:
# let's train the model
learning_rate = 0.01
n_epochs = 20

for epoch in range(n_epochs):
  y_pred = X_train @ weights + bias
  loss = ((y_pred - y_train)**2).mean()
  loss.backward()
  with torch.no_grad():
    bias -= learning_rate * bias.grad
    weights -= learning_rate * weights.grad
    bias.grad.zero_()
    weights.grad.zero_()
  print(f"Epoch {epoch + 1}/{n_epochs}, Loss {loss.item()}")

Epoch 1/20, Loss 6.800904273986816
Epoch 2/20, Loss 6.542640209197998
Epoch 3/20, Loss 6.295825481414795
Epoch 4/20, Loss 6.059910774230957
Epoch 5/20, Loss 5.834374904632568
Epoch 6/20, Loss 5.6187262535095215
Epoch 7/20, Loss 5.412496089935303
Epoch 8/20, Loss 5.215242862701416
Epoch 9/20, Loss 5.026545524597168
Epoch 10/20, Loss 4.846004486083984
Epoch 11/20, Loss 4.6732401847839355
Epoch 12/20, Loss 4.507894992828369
Epoch 13/20, Loss 4.349626064300537
Epoch 14/20, Loss 4.198109149932861
Epoch 15/20, Loss 4.053036212921143
Epoch 16/20, Loss 3.914114236831665
Epoch 17/20, Loss 3.781064033508301
Epoch 18/20, Loss 3.653620958328247
Epoch 19/20, Loss 3.53153395652771
Epoch 20/20, Loss 3.414562463760376


In [72]:
X_new = X_test[:3]
with torch.no_grad():
  y_pred = X_new @ weights + bias

In [73]:
y_pred

tensor([[-0.7982],
        [ 0.3902],
        [ 1.6807]])

Linear Regression with torch high level API

In [90]:
import torch.nn as nn
model = nn.Linear(in_features= n_features, out_features = 1)

In [91]:
# parameters
model.bias, model.weight

(Parameter containing:
 tensor([0.2229], requires_grad=True),
 Parameter containing:
 tensor([[-0.1924,  0.1884, -0.1210,  0.0905, -0.2126, -0.1224, -0.0620, -0.2207]],
        requires_grad=True))

In [92]:
for param in model.parameters():
  print(param)
for named_param in model.named_parameters():
  print(named_param[0], named_param[1])

Parameter containing:
tensor([[-0.1924,  0.1884, -0.1210,  0.0905, -0.2126, -0.1224, -0.0620, -0.2207]],
       requires_grad=True)
Parameter containing:
tensor([0.2229], requires_grad=True)
weight Parameter containing:
tensor([[-0.1924,  0.1884, -0.1210,  0.0905, -0.2126, -0.1224, -0.0620, -0.2207]],
       requires_grad=True)
bias Parameter containing:
tensor([0.2229], requires_grad=True)


In [93]:
# here model is just normal.
# so model is not train yet. it will make terriable predications
# Note: when call the this torch will internall call forward() method Which is just as see in previous example. X @ self.weight + self.bias
model(X_train[:3])

tensor([[ 0.4945],
        [-0.9500],
        [ 0.6535]], grad_fn=<AddmmBackward0>)

In [94]:
# optimizer: In short it updates the learnable parameters values like weights and bias. You can choose different kinds algorithm to perform backpropagation
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()

In [96]:
def train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs):
  for epoch in range(n_epochs):
    y_pred = model(X_train)
    loss = criterion(y_pred, y_train)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    print(f'{epoch}/{n_epochs}, loss: {loss.item()}')

In [97]:
train_bgd(model, optimizer, mse, X_train, y_train, 20)

0/20, loss: 5.261836528778076
1/20, loss: 5.0687665939331055
2/20, loss: 4.883979320526123
3/20, loss: 4.707101345062256
4/20, loss: 4.537777423858643
5/20, loss: 4.37566614151001
6/20, loss: 4.220447540283203
7/20, loss: 4.071813583374023
8/20, loss: 3.9294722080230713
9/20, loss: 3.7931437492370605
10/20, loss: 3.6625633239746094
11/20, loss: 3.53747820854187
12/20, loss: 3.4176461696624756
13/20, loss: 3.302838087081909
14/20, loss: 3.19283390045166
15/20, loss: 3.0874252319335938
16/20, loss: 2.9864115715026855
17/20, loss: 2.889603853225708
18/20, loss: 2.796818494796753
19/20, loss: 2.707883596420288


array([[12, 12, 12],
       [12, 23, 43],
       [12, 34, 34]])